# An age-stratified SEIRS model, calibrated

This case study builds one model end to end and fits it. The model has

- **four states** — susceptible, exposed, infectious, recovered, with waning
  immunity back to susceptible;
- **three age bands**, with an ageing population that settles into a
  skewed-old steady state;
- **age mixing** through a contact matrix, starting homogeneous;
- **age-varying infectiousness**, as a parameter;
- **no infection in the initial population** — the epidemic arrives through a
  parameterised importation inflow.

From there we generate outputs at a chosen set of parameters, add noise, thin
them to a weekly sample, and calibrate back against those noisy sparse targets.

Everything here uses features summer4 already ships. Three pieces are written
by hand rather than reached for as primitives, and it is worth knowing which
before you start: the **force of infection**, the **mixing matrix**, and the
**age-varying infectiousness** are all plain JAX inside a `derived_fn` hook.
summer4 has no force-of-infection constructor, no `set_mixing_matrix`, and no
infectiousness adjustments — those are ledger rows `F7`, `F8`, `M1` and `A4`,
and they are the subject of work package WP6. What follows is the honest amount
of code that costs you today, which is about fifteen lines.

The one substantive modelling result falls out of that arrangement: a
**homogeneous mixing matrix cannot identify age-varying infectiousness**,
because it collapses the force of infection to a single number shared by every
band. We measure that rather than assert it, and then fix it.

In [ ]:
from datetime import date
from typing import Any, Callable, NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
import optax
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    ComputedValue,
    Dest,
    EntryFlow,
    Epoch,
    Everything,
    ExitFlow,
    FlowMass,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TraitChain,
    TransitionFlow,
    derived_refs,
)

pd.options.plotting.backend = "plotly"
# "notebook_connected" emits a small text/html payload that loads plotly.js from
# a CDN, which is what the Sphinx build can render. The default mimetype
# renderer produces output myst-nb shows as a blank placeholder.
pio.renderers.default = "notebook_connected"

YEAR = 365.0

state = Property("state", ("S", "E", "I", "R"))
age = Property("age", ("0-14", "15-64", "65+"))

## The compartment space

Two properties crossed: `state` and `age`. Twelve compartments, and no ragged
edges — every state exists in every band.

In [ ]:
pmap = PropertyMap.from_property(state).stratify(age)

assert pmap.size == 12
print(pmap.labels()[:3], "...", pmap.labels()[-1:])

## An ageing population that skews old

Ageing is one named flow. `TraitChain` carries the whole chain of bands in a
single declaration, with a per-band rate equal to the reciprocal of the band's
width, so people spend on average fifteen years in `0-14` and fifty in
`15-64`. The oldest band is terminal: it receives people and loses them only to
death.

Note the units. Progression and recovery below are per **day**, so the ageing
and death rates are per day too.

In [ ]:
AGEING = (1.0 / (15.0 * YEAR), 1.0 / (50.0 * YEAR))
DEATH = 1.0 / (80.0 * YEAR)

ageing = TraitChain(age, (("0-14", "15-64"), ("15-64", "65+")), rates=AGEING)

# One pair per boundary between bands: "65+" is terminal and has no outgoing edge.
assert len(ageing.pairs) == len(age.traits) - 1

## Mixing, infectiousness, and the force of infection

This is the hand-written part. `derived_fn` is called on every evaluation of
the vector field, receives the bare compartment vector and the time, and
returns a struct that flow rates can point at through `derived_refs`.

The force of infection for band $a$ is

$$\lambda_a(t) = c \sum_b K_{ab}\, \nu_b \frac{I_b(t)}{N_b(t)}$$

for contact rate $c$, mixing matrix $K$, and per-band infectiousness $\nu_b$.
Two mechanics make it work:

- **`broadcast_over`** is the bridge. `derived_fn` computes a three-element
  vector, one per age band, but a flow rate is applied per compartment.
  `pd_y.broadcast_over(age, foi_by_age)` expands it to all twelve rows, and
  rate alignment then gathers the right element for each edge. A bare
  three-element array would *not* work: it would be silently misread as a
  per-edge rate whenever the flow happens to have three edges.
- **`where` replaces, it does not keep.** `pd_y.where(sel, 0.0)` zeroes the
  compartments that `sel` matches, like `Series.mask` rather than
  `Series.where`. Infectious prevalence is therefore
  `where(~state["I"], 0.0)`. Reading it the other way round gives a force of
  infection over $S + E + R$; the model still integrates, and the epidemic
  simply saturates on the first day.

The mixing matrix lives in `params` rather than in a closure, so we can change
it later without rebuilding the model.

In [ ]:
class Derived(NamedTuple):
    """What `derived_fn` returns every time the vector field is evaluated."""

    foi: Any
    importation: Any


refs = derived_refs(Derived)


def derived_fn(params: Any, *, y: Any, t: Any) -> Derived:
    pd_y = PropertyData(pmap, y)
    n_by_age = pd_y.sum_over(age).data
    # `where` REPLACES the compartments a selector matches, so `~state["I"]`
    # is what *keeps* the infectious ones. Reading it as "keep where" silently
    # gives a force of infection over S + E + R.
    i_by_age = pd_y.where(~state["I"], 0.0).sum_over(age).data
    shedding = (jnp.asarray(params["infectiousness"]) * i_by_age) / n_by_age
    foi_by_age = params["contact_rate"] * (jnp.asarray(params["mixing"]) @ shedding)
    pulse = jnp.exp(-0.5 * ((t - params["seed_time"]) / params["seed_width"]) ** 2)
    return Derived(
        foi=pd_y.broadcast_over(age, foi_by_age),
        importation=params["seed_rate"] * pulse,
    )

## Assembling the model

Eight flows. Three of them are worth a second look:

- **`ageing`** uses the `TraitChain` pairing declared above, so one flow
  becomes eight edges — two band boundaries times four states.
- **`birth`** takes its rate from `death.sum()`. Flow rates can reference other
  flows, and `compile()` topologically sorts them, so births exactly replace
  deaths and the total population is conserved.
- **`importation`** is an absolute inflow into the infectious compartments,
  split across bands. Its rate is a derived value, so it can vary with time.

In [ ]:
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["E"], refs.foi))
model.add_flow(TransitionFlow("progression", state["E"], state["I"], 1.0 / 5.0))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 10.0))
model.add_flow(TransitionFlow("waning", state["R"], state["S"], 1.0 / 180.0))
model.add_flow(TransitionFlow("ageing", age.present(), age.present(), 1.0, pairing=ageing))
death = model.add_flow(ExitFlow("death", Everything(), DEATH))
model.add_flow(EntryFlow("birth", state["S"] & age["0-14"], death.sum()))
model.add_flow(
    EntryFlow(
        "importation",
        state["I"],
        refs.importation,
        split={age: {"0-14": 0.1, "15-64": 0.7, "65+": 0.2}},
    )
)

compiled = model.compile(derived_fn=derived_fn)
print(compiled.order)

## Where the initial population comes from

We need a population that skews old, with nobody infected. Rather than invent
one, run the demography on its own: switch transmission off, start from a
young population, and integrate for three hundred years.

This is also the answer to a question the coverage ledger makes look harder
than it is. Rows `L4`, `L5` and `S8` are marked `none` because there is no
declarative `set_initial_population`. But `run()` takes the initial state as
its second positional argument, and `PropertyData.at[...]` builds one from a
selector, so an arbitrary initial population has always been available.

In [ ]:
HOMOGENEOUS = np.ones((3, 3)) / 3.0

QUIET = {
    "contact_rate": 0.0,
    "infectiousness": jnp.ones(3),
    "mixing": HOMOGENEOUS,
    "seed_rate": 0.0,
    "seed_time": 0.0,
    "seed_width": 1.0,
}

start_young = (
    PropertyData.wrap(pmap, jnp.zeros(pmap.size))
    .at[state["S"]]
    .set(jnp.array([500_000.0, 400_000.0, 100_000.0]))
)

demography = compiled.run(
    QUIET,
    start_young,
    t0=0.0,
    t1=300 * YEAR,
    dt=YEAR,
    save=SavePlan(
        requests={
            "by_age": SaveRequest(Compartments(sum_over=age)),
            "compartments": SaveRequest(Compartments()),
        }
    ),
    solver="tsit5",
    rtol=1e-8,
    atol=1e-8,
)

structure = np.asarray(demography["by_age"].values.data)
shares = structure[-1] / structure[-1].sum()
print("age shares after 300 years:", dict(zip(age.traits, shares.round(4))))

A three-band population with ageing rates $a_1, a_2$ and uniform death rate
$\mu$ has a steady state we can write down, so the run is checkable rather
than merely plausible:

$$N_1 \propto \frac{1}{a_1 + \mu}, \quad
  N_2 \propto \frac{a_1 N_1}{a_2 + \mu}, \quad
  N_3 \propto \frac{a_2 N_2}{\mu}$$

In [ ]:
a_young, a_mid = AGEING
n_young = 1.0 / (a_young + DEATH)
n_mid = a_young * n_young / (a_mid + DEATH)
n_old = a_mid * n_mid / DEATH
analytic = np.array([n_young, n_mid, n_old]) / (n_young + n_mid + n_old)

np.testing.assert_allclose(shares, analytic, rtol=1e-3)
assert shares[-1] > 0.5
np.testing.assert_allclose(structure[-1].sum(), 1_000_000.0, rtol=1e-6)
print("analytic steady state:", analytic.round(4))

Over three centuries the young population ages into its steady state, and
just over half of it ends up in the oldest band.

In [ ]:
profile = pd.DataFrame(structure / structure.sum(axis=1, keepdims=True), columns=list(age.traits))
profile.index = np.asarray(demography["by_age"].times.values) / YEAR
profile.index.name = "year"

figure = profile.plot(title="An ageing population: age structure settling into a skewed-old state")
figure.update_layout(yaxis_title="share of population", legend_title="age band")
figure

The last row of the demographic run is the initial population for the
epidemic. Nobody is infectious, because nothing ever introduced infection.

In [ ]:
y0 = PropertyData.wrap(pmap, jnp.asarray(np.asarray(demography["compartments"].values.data)[-1]))

infected_at_start = float(jnp.sum(y0.data[pmap.select(state["I"])]))
assert infected_at_start == 0.0
print(f"infectious people at t0: {infected_at_start}")
print("susceptible by age:", np.asarray(y0.data[pmap.select(state["S"])]).round(0))

## Target parameters

Infectiousness is parameterised per band, but a scale shared with
`contact_rate` would be redundant: doubling every weight and halving the
contact rate gives exactly the same model. Pinning the population-weighted
mean to one removes that redundancy, so the weights describe only the *shape*
of the age gradient.

In [ ]:
POPULATION_SHARE = jnp.asarray(shares)


def normalise(weights: Any) -> Any:
    """Scale infectiousness so its population-weighted mean is exactly 1."""
    return weights / jnp.sum(POPULATION_SHARE * weights)


TARGET_PARAMS = {
    "contact_rate": 0.55,
    "infectiousness": normalise(jnp.array([0.7, 1.0, 1.4])),
    "mixing": HOMOGENEOUS,
    "seed_rate": 5.0,
    "seed_time": 30.0,
    "seed_width": 6.0,
}

np.testing.assert_allclose(
    float(jnp.sum(POPULATION_SHARE * TARGET_PARAMS["infectiousness"])), 1.0, rtol=1e-6
)
print("infectiousness weights:", np.asarray(TARGET_PARAMS["infectiousness"]).round(4))

## Outputs at the target parameters

Two quantities per band. Prevalence is a compartment query; notifications are
progressions into `I`, which is a flow query — and `where=Dest(age[band])`
picks the single edge entering that band, giving one column rather than a
grouped total. That detail matters later: targets reshape onto a
one-dimensional series, and a `sum_over` grouped output would not fit.

In [ ]:
HORIZON = 250.0
EPOCH = Epoch(date(2020, 1, 1))
TSIT5 = {"solver": "tsit5", "rtol": 1e-7, "atol": 1e-7}


def notifications(band: str) -> FlowMass:
    """Progressions into I inside one age band: one edge, so one column."""
    return FlowMass("progression", where=Dest(age[band]))


REPORTING_PLAN = SavePlan(
    requests={
        **{f"notifications_{b}": SaveRequest(notifications(b)) for b in age.traits},
        **{
            f"prevalence_{b}": SaveRequest(Compartments(where=state["I"] & age[b]))
            for b in age.traits
        },
    }
)


def simulate(params: Any, plan: SavePlan = REPORTING_PLAN) -> Any:
    """Integrate to the horizon on a daily grid, dated by the epoch."""
    return compiled.run(params, y0, t0=0.0, t1=HORIZON, dt=1.0, save=plan, epoch=EPOCH, **TSIT5)


truth = simulate(TARGET_PARAMS)

for band in age.traits:
    assert truth[f"notifications_{band}"].values.data.shape == (int(HORIZON) + 1, 1)
peaks = {b: round(float(jnp.max(truth[f"prevalence_{b}"].values.data))) for b in age.traits}
print("peak prevalence:", peaks)

Under homogeneous mixing every band sees the same force of infection, so the
three curves differ only through how many susceptible people each band holds
and how fast they are depleted. The oldest band dominates because it is the
largest.

In [ ]:
def band_frame(prefix: str, result: Any) -> pd.DataFrame:
    """One column per age band, indexed by calendar date."""
    columns = {b: np.asarray(result[f"{prefix}_{b}"].values.data).reshape(-1) for b in age.traits}
    frame = pd.DataFrame(columns)
    frame.index = result[f"{prefix}_{age.traits[0]}"].times.as_dates()
    frame.index.name = "date"
    return frame


figure = band_frame("notifications", truth).plot(
    title="Daily progressions to infectious, by age band, at the target parameters"
)
figure.update_layout(yaxis_title="notifications per day", legend_title="age band")
figure

## Noisy, sparse observations

Real data arrive on scattered days with error. We sample weekly between days 21
and 175, multiply by lognormal noise with a 10% log standard deviation, and
declare one `Target` per band. `TargetSet.plan` folds the observation times
into a `SavePlan`, so a likelihood run never materialises a dense trajectory.

In [ ]:
OBSERVED_TIMES = np.arange(21.0, 176.0, 7.0)
NOISE = 0.10


def observe(result: Any, seed: int = 20200101) -> tuple[dict[str, Any], TargetSet]:
    """Sample weekly, multiply by lognormal noise, and declare the targets."""
    rng = np.random.default_rng(seed)
    values = {}
    for band in age.traits:
        exact = np.asarray(
            result[f"notifications_{band}"].at_times(OBSERVED_TIMES).values.data
        ).reshape(-1)
        values[band] = exact * np.exp(rng.normal(0.0, NOISE, OBSERVED_TIMES.size))
    targets = TargetSet(
        tuple(
            Target(
                key=f"notifications_{band}",
                times=OBSERVED_TIMES,
                values=values[band],
                quantity=notifications(band),
            )
            for band in age.traits
        )
    )
    return values, targets


observations, targets = observe(truth)
likelihood_plan = targets.plan(SavePlan())

for key in targets.keys():
    assert likelihood_plan.requests[key].ts.size == OBSERVED_TIMES.size
print(f"{OBSERVED_TIMES.size} weekly observations in each of {len(targets.keys())} bands")

Because all three targets share the same observation times, they collapse into
a single solver save group — one `SubSaveAt`, not three. And the likelihood
plan solves for an order of magnitude fewer rows than a daily grid would.

In [ ]:
from summer4.results.groups import group_requests

groups = group_requests(compiled.expand(likelihood_plan), default_ts=np.array([0.0]))
assert len(groups) == 1
assert set(groups[0].keys) == set(targets.keys())

sparse_rows = sum(request.ts.size for request in likelihood_plan.requests.values())
dense_rows = len(targets.keys()) * (int(HORIZON) + 1)
print(f"rows the likelihood solves for: {sparse_rows}, against {dense_rows} on a daily grid")
assert sparse_rows * 5 < dense_rows

## A loss on the log scale

`TargetSet.residuals` subtracts on the raw scale. That is the wrong geometry
here: notification counts span three orders of magnitude across the wave, so a
raw-scale sum of squares is dominated by the peak and is badly conditioned —
Adam stalls on it. The noise was injected multiplicatively, so least squares on
the **log** scale is the estimator the noise model implies.

`residuals` cannot do that for us, so we use `TargetSet.gather` instead, which
returns the predictions at the observation times and leaves the comparison to
us. That is the shape of the gap WP10 fills: the target machinery lines the
times up, and the likelihood is still yours to write.

In [ ]:
figure = band_frame("notifications", truth).plot(
    title="Noisy weekly observations against the curves that generated them"
)
for band in age.traits:
    figure.add_scatter(
        x=EPOCH.from_model(OBSERVED_TIMES),
        y=observations[band],
        mode="markers",
        name=f"observed {band}",
    )
figure.update_layout(yaxis_title="notifications per day", legend_title="series")
figure

## Calibrating what the data identify

Start with the two parameters that set the size and timing of the wave, holding
the infectiousness gradient at its known value. `run` is called *inside* the
loss — a finished `Result` cannot be passed across a `jit` boundary, because
its time axes would arrive as tracers.

In [ ]:
def log_least_squares(targets: TargetSet, observations: dict[str, Any]) -> Callable[[Any], Any]:
    """Least squares on the log scale, which is the estimator the noise implies."""
    log_observed = {f"notifications_{b}": jnp.log(jnp.asarray(v)) for b, v in observations.items()}
    plan = targets.plan(SavePlan())

    def loss(params: Any) -> Any:
        result = compiled.run(params, y0, t0=0.0, t1=HORIZON, dt=1.0, save=plan, **TSIT5)
        predicted = targets.gather(result)
        return sum(
            jnp.mean((jnp.log(predicted[k].values.data.reshape(-1)) - v) ** 2)
            for k, v in log_observed.items()
        )

    return loss


objective = log_least_squares(targets, observations)
print(f"loss at the target parameters: {float(objective(TARGET_PARAMS)):.5f}")
print(f"variance of the injected noise: {NOISE ** 2:.5f} per band")

Both recover to within a few percent, from a starting point nearly a factor of
two away, in a few hundred iterations.

In [ ]:
def fit(
    loss: Callable[[Any], Any], start: dict[str, Any], steps: int = 600, rate: float = 0.05
) -> tuple[dict[str, Any], list[float]]:
    """Adam on a jitted value_and_grad straight through the ODE solve."""
    free = dict(start)
    step = jax.jit(jax.value_and_grad(loss))
    optimiser = optax.adam(rate)
    opt_state = optimiser.init(free)
    history = []
    for _ in range(steps):
        value, gradient = step(free)
        history.append(float(value))
        updates, opt_state = optimiser.update(gradient, opt_state)
        free = optax.apply_updates(free, updates)
    return free, history


def with_intensity(free: Any, base: Any) -> Any:
    """contact_rate and seed_rate only; the infectiousness weights stay known."""
    return {
        **base,
        "contact_rate": jnp.exp(free["log_contact"]),
        "seed_rate": jnp.exp(free["log_seed"]),
    }


START = {"log_contact": jnp.log(0.3), "log_seed": jnp.log(1.0)}
free, history = fit(lambda f: objective(with_intensity(f, TARGET_PARAMS)), START)
intensity_fit = with_intensity(free, TARGET_PARAMS)

print(f"contact_rate {float(intensity_fit['contact_rate']):.3f}  target 0.550")
print(f"seed_rate    {float(intensity_fit['seed_rate']):.3f}  target 5.000")
print(f"loss {history[-1]:.5f}, down from {history[0]:.5f}")

assert abs(float(intensity_fit["contact_rate"]) / TARGET_PARAMS["contact_rate"] - 1.0) < 0.05
assert abs(float(intensity_fit["seed_rate"]) / TARGET_PARAMS["seed_rate"] - 1.0) < 0.15

## What homogeneous mixing cannot tell you

Now free the infectiousness gradient as well and refit.

In [ ]:
figure = pd.Series(history, name="loss").plot(
    title="Adam on a jitted value_and_grad through the solve", log_y=True
)
figure.update_layout(xaxis_title="iteration", yaxis_title="mean squared log residual")
figure

The gradient comes back **inverted** — infectiousness decreasing with age,
where the data were generated with it increasing — and it fits at least as well
as the parameters that generated the data. This is not an optimiser failure. It
is the model telling us the data do not identify those weights.

The reason is visible if we capture the force of infection itself. A
`ComputedValue` request saves any path in the `derived_fn` return struct, so we
can look at $\lambda_a$ directly.

In [ ]:
def with_weights(free: Any, base: Any) -> Any:
    """Also free the outer two infectiousness weights; the middle band anchors them."""
    weights = jnp.stack([jnp.exp(free["log_young"]), jnp.array(1.0), jnp.exp(free["log_old"])])
    return {**with_intensity(free, base), "infectiousness": normalise(weights)}


START_WEIGHTS = {**START, "log_young": jnp.array(0.0), "log_old": jnp.array(0.0)}
free, _ = fit(lambda f: objective(with_weights(f, TARGET_PARAMS)), START_WEIGHTS)
homogeneous_fit = with_weights(free, TARGET_PARAMS)

recovered = np.asarray(homogeneous_fit["infectiousness"])
print("recovered infectiousness:", recovered.round(3))
print("target infectiousness:   ", np.asarray(TARGET_PARAMS["infectiousness"]).round(3))
print(
    f"loss {float(objective(homogeneous_fit)):.5f} vs {float(objective(TARGET_PARAMS)):.5f} at target"
)

# The age gradient comes back inverted, and fits at least as well as the truth.
assert recovered[0] > recovered[-1]
assert float(objective(homogeneous_fit)) <= float(objective(TARGET_PARAMS))

Zero. Under homogeneous mixing every band experiences *exactly* the same force
of infection, so the infectiousness weights enter the model only through a
single scalar sum. Many different gradients produce the same sum, and the
per-band data cannot distinguish them.

In [ ]:
foi_plan = SavePlan(requests={"foi": SaveRequest(ComputedValue(path=("foi",)))})
susceptible = pmap.select(state["S"])


def foi_by_band(params: Any) -> np.ndarray:
    # A ComputedValue output carries the raw array, not a PropertyData.
    captured = simulate(params, foi_plan)["foi"]
    assert captured.dims == ("time",)
    return np.asarray(captured.values)[:, susceptible]


homogeneous_foi = foi_by_band(TARGET_PARAMS)
print("force of infection by band at t=60:", homogeneous_foi[60].round(6))
spread = float(np.abs(homogeneous_foi - homogeneous_foi[:, :1]).max())
print(f"largest difference between bands, over all times: {spread:.3e}")
assert spread == 0.0

## Assortative mixing restores the age gradient

Mixing matters precisely because it breaks that degeneracy. Swap in a matrix
where people contact their own age band most — rows still summing to one, so
`contact_rate` keeps its meaning — and the force of infection becomes
band-specific.

This is a one-line change and no new model code, because the matrix is a
parameter. (It is also not a *reciprocal* matrix: a real contact matrix
satisfies $K_{ab} N_a = K_{ba} N_b$, and nothing here enforces that. Checking
and rescaling empirical matrices is WP9.)

In [ ]:
ASSORTATIVE = np.array([[0.6, 0.3, 0.1], [0.2, 0.6, 0.2], [0.1, 0.3, 0.6]])
np.testing.assert_allclose(ASSORTATIVE.sum(axis=1), 1.0)

ASSORTATIVE_PARAMS = {**TARGET_PARAMS, "mixing": ASSORTATIVE}

assortative_foi = foi_by_band(ASSORTATIVE_PARAMS)
print("force of infection by band at t=60:", assortative_foi[60].round(6))
assert np.abs(assortative_foi - assortative_foi[:, :1]).max() > 1e-3

With band-specific exposure the age gradient is identified: refitting the same
four parameters against data generated under assortative mixing recovers a
monotone increasing gradient, and the contact rate along with it.

In [ ]:
assortative_truth = simulate(ASSORTATIVE_PARAMS)
assortative_observations, assortative_targets = observe(assortative_truth)
assortative_objective = log_least_squares(assortative_targets, assortative_observations)

free, _ = fit(lambda f: assortative_objective(with_weights(f, ASSORTATIVE_PARAMS)), START_WEIGHTS)
assortative_fit = with_weights(free, ASSORTATIVE_PARAMS)

recovered = np.asarray(assortative_fit["infectiousness"])
print("recovered infectiousness:", recovered.round(3))
print("target infectiousness:   ", np.asarray(TARGET_PARAMS["infectiousness"]).round(3))
print(f"contact_rate {float(assortative_fit['contact_rate']):.3f}  target 0.550")

# The gradient is the right way round, and monotone, as the data were made.
assert recovered[0] < recovered[1] < recovered[2]
assert abs(float(assortative_fit["contact_rate"]) / TARGET_PARAMS["contact_rate"] - 1.0) < 0.10

## Reporting run

A likelihood plan and a reporting plan are two `SavePlan`s, not two modes of one
model. Having fitted on 69 rows, re-run on the dense grid for the figures.

In [ ]:
report = simulate(intensity_fit)

figure = band_frame("notifications", report).plot(
    title="Calibrated model against the observations it was fitted to"
)
for band in age.traits:
    figure.add_scatter(
        x=EPOCH.from_model(OBSERVED_TIMES),
        y=observations[band],
        mode="markers",
        name=f"observed {band}",
    )
figure.update_layout(yaxis_title="notifications per day", legend_title="series")
figure

Finally, the query surface the results layer provides: calendar resampling,
cumulative sums, and `gather`, which agrees with an explicit `at_times` because
it is the same interpolation underneath.

In [ ]:
weekly = report["notifications_65+"].resample("W", how="sum")
cumulative = report["notifications_65+"].cumulative()

print(f"weekly rows: {weekly.values.data.shape[0]}, from {int(HORIZON) + 1} daily rows")
total_old = float(np.asarray(cumulative.values.data).reshape(-1)[-1])
print(f"cumulative notifications in 65+: {total_old:,.0f}")

np.testing.assert_allclose(
    np.asarray(targets.gather(report)["notifications_65+"].values.data).reshape(-1),
    np.asarray(report["notifications_65+"].at_times(OBSERVED_TIMES).values.data).reshape(-1),
    rtol=1e-5,
)
assert weekly.values.data.shape[0] < int(HORIZON) + 1

## What this needed that summer4 does not provide

The model, the outputs, the sparse targets and the fit are all shipped
features. Four things were hand-written, and each is a known work package:

| Hand-written here | Ledger | Package |
|---|---|---|
| Force of infection as `contact * I / N` | `F7`, `F8` `partial` | WP6 |
| Mixing matrix as a `params` array | `M1` `none` | WP6 |
| Age-varying infectiousness | `A4` `none` | WP6 |
| Initial population via `run(params, y0, ...)` | `L4`, `L5`, `S8` `none` | WP3 |
| Gaussian importation pulse in JAX | `P6`-`P8` `none` | WP5 |
| The loss itself, and any likelihood | — | WP10 |

Four sharp edges cost real time while writing this, and are recorded in
[`futureplans/`](https://github.com/monash-emu/summer4/tree/main/futureplans):
`PropertyData.where` has mask polarity under a name that reads as keep;
`CompiledModel.describe` cannot size a plan for a model whose `derived_fn`
indexes `params`, because it passes `None`; `TargetSet.residuals` reshapes but
cannot reduce; and `Output.plot` hardcodes matplotlib.

For the assessment this case study was written to support, with every
requirement mapped to a mechanism and a ledger row, see
{doc}`../evaluation/age-stratified-seirs-case-study`.